# N006 · 字符串与基础解析

**目标：** 在字符和字符串层面完成基础转换。

**先修：** N005。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 不可变性；split/join；ord/chr；字符分类；Unicode 与题目字符集。

**配套讲解来源：** [同名逐章意见](../../comment/006_strings_unicode_parsing.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章解决两道字符串入门题，核心训练的是"把字符串当成不可变的字符序列来扫描和重组"。

**问题 1（IP 地址脱敏）**：给你一个 IPv4 地址字符串，把里面每个点 `.` 换成 `[.]`。具体例子：输入 `'1.1.1.1'`，输出 `'1[.]1[.]1[.]1'`——四个点各被换成三个字符的 `[.]`，数字部分原样保留。这是一次"局部替换"：扫到点就换成指定串，其他字符照抄。

**问题 2（Goal Parser 解释器）**：给定一条命令字符串 command，它是三种"合法 token"的串联：`G` 翻译成 `G`、`()` 翻译成 `o`、`(al)` 翻译成 `al`。要求输出整体翻译结果。具体例子：输入 `'G()(al)'`，输出 `'Goal'`——拆开看是 `G` + `()` + `(al)` 三个 token，分别翻译成 `G`、`o`、`al`，拼起来就是 `Goal`。这题的精髓在于：扫描时索引必须**一次跳过整个 token**，比如吃掉 `(al)` 就要前进 4 格，绝不能吃掉 `(` 之后又把 `al)` 当新东西重新识别。本章还明确了契约：输入只允许这三种 token 的串联，遇到别的（比如 `(bad)`）要显式抛错，而不是默默输出错误结果。

## 2. 基础知识：先读懂这些词

- **不可变性（immutability）**：字符串一旦创建就不能被修改。`s[0]='x'` 会直接报错；所有"修改"操作（replace、upper 等）其实都是返回一个**新字符串**，原串不变。这与上一章可变的列表形成对照。
- **token（词法单元）**：把长字符串切分成的一个个有意义的片段。本章的 token 就是 `G`、`()`、`(al)` 三种，解析 = 从左到右把输入切成 token 再逐个翻译。
- **`startswith(前缀, 起点)`**：检查字符串在指定起点处是否以某前缀开头，返回 True/False。`command.startswith('()',i)` 读作"从第 i 个字符开始，后面是不是正好排着 `(` 和 `)`"。它是无副作用的" lookahead（往前看一眼）"，不移动指针。
- **扫描指针（i）**：一个记录"当前处理到哪里"的下标变量。指针左边的字符已经消费完，右边的还没碰。每识别一个 token，i 就跳过这个 token 的长度。
- **split / join**：`split` 按分隔符把字符串拆成列表；`join` 反过来，把一堆字符串用指定粘合剂连成一个。`''.join(pieces)` 就是用**空字符串**当粘合剂，把片段列表无缝拼起来。join 的好处是只分配一次结果空间，比循环里 `+=` 反复复制高效。
- **ord / chr**：`ord('A')` 给字符的编码数字（Unicode 码位），`chr(65)` 反过来把编码变回字符。很多字符计算题（比如字母移位）靠这一对函数。本章代码没直接用到，但它是字符串知识点的标配。
- **字符分类**：判断字符是大写、小写、数字之类，用 `isupper()`、`isdigit()` 这类方法，属于"看一眼字符属于哪类"的工具箱。
- **Unicode 与题目字符集**：Python 字符串按 Unicode 处理，能装中文、emoji；但算法题通常明确限定字符集（比如"仅小写字母"），解析时按题目限定的字符集来，别替题目加戏。本章就限定死了三种 token。
- **fail fast（快速失败）**：遇到不符合契约的输入立刻抛异常，而不是给出貌似正常的错误输出。`raise ValueError(f'invalid token at index {i}')` 就是这个原则。

## 3. 思路推导：从小例子开始

**思路 A：IP 脱敏。**

- **Step 1：识别问题类型。** "把某个子串统一换成另一个子串"是标准替换，Python 内置 `replace` 一步到位。
- **Step 2：调用 `address.replace('.', '[.]')`。** 它会扫描整个 address，把每个点换成 `[.]`，并返回新字符串（原串不可变也不需要变）。
- **Step 3：验证。** `'1.1.1.1'` 原长 7 个字符（4 个数字 + 3 个点），替换后每个点变成 3 个字符，新长度 = 4 + 3×3 = 13，结果 `1[.]1[.]1[.]1`。

**手算演示一轮（'1.1.1.1'）**：replace 从左扫到右：遇到 `1` 不匹配点，照抄；遇到 `.` 匹配，产出 `[.]`；再 `1`、再 `[.]`……最终新字符串按顺序是 `1` `[.]` `1` `[.]` `1` `[.]` `1`，拼起来 `1[.]1[.]1[.]1`。全程原 address 没有被修改（它不可变），我们拿到的是新对象。

**思路 B：Goal Parser。**

- **Step 1：确定 token 表。** 合法 token 只有三种：`G`（长 1）、`()`（长 2）、`(al)`（长 4）。它们的翻译是固定的：`G→G`、`()→o`、`(al)→al`。
- **Step 2：从左到右扫描，逐个"认领"。** 用指针 i 从 0 开始，每轮依次问三个问题：从 i 开始是 `G` 吗？是 `()` 吗？是 `(al)` 吗？注意先问谁有讲究——`G` 与另外两个开头不同（`(`），而 `()` 与 `(al)` 都以 `(` 开头但第二个字符不同，三种起始结构可区分，顺序不影响正确性。
- **Step 3：吃掉 token 就跳整段。** 认出 `G` 就 append 'G' 且 i 前进 1；认出 `()` 就 append 'o' 且 i 前进 2；认出 `(al)` 就 append 'al' 且 i 前进 4。**前进的格数必须等于 token 长度**，多一格会把后面的字符漏掉，少一格会把 token 的尾巴误当新 token。
- **Step 4：三个都不匹配就抛错。** 输入不符合"三种 token 串联"的契约（比如 `(bad)`），直接 `raise ValueError` 并带上出错位置。
- **Step 5：最后用 join 拼接。** 全部 token 翻译完，用 `''.join(pieces)` 一次性拼出答案。

**手算演示一轮（'G()(al)'）**：初始 i=0。第 1 轮：从 0 开始是 `G`，append 'G'，i 跳到 1。第 2 轮：从 1 开始不是 `G`，但是 `()`，append 'o'，i 跳到 3。第 3 轮：从 3 开始是 `(al)`，append 'al'，i 跳到 7，恰好等于 len(command)=7，循环结束。pieces = `['G','o','al']`，join 得 `'Goal'`。

**小实例表格（“运行示例”部分）**：“运行示例”部分 用 `show_table` 列了三行——`'G'` 翻译成 `'G'` 消费 1 个字符；`'()'` 翻译成 `'o'` 消费 2 个字符；`'(al)'` 翻译成 `'al'` 消费 4 个字符。这张表就是 token 表的实物版：每个 token 对应固定输出和固定步长。

## 4. 核心代码：defang_ip

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def defang_ip(address: str) -> str:
    return address.replace('.', '[.]')
```

### 逐段读懂代码

**核心实现（“分段实现”部分）对应实现：**

（对应程序见下方分段实现与完整代码。）

**函数一：`defang_ip`**

- `return address.replace('.', '[.]')` —— 一行完成。`replace(旧, 新)` 把字符串里所有出现的旧子串换成新子串并返回新字符串。因为字符串不可变，replace 不可能"原地"修改 address，它必然造新串返回，天然满足"不动输入"的契约。
- 补充一个 replace 的语义细节：它替换的是**所有**出现的匹配子串，不是只换第一个，`'1.1.1.1'` 里三个点全部中招；如果输入里一个点都没有（比如 `'255'`），replace 找不到匹配，返回一个内容与原串相等的字符串。“自动测试”部分 没有单独测"无点"输入，你做自测时可以补一条 `defang_ip('255') == '255'`。

**函数二：`interpret`**

- `pieces=[]` —— 准备一个空列表收集每个 token 的翻译片段。先攒片段、最后统一 join，是字符串构建的标准姿势。
- `i=0` —— 扫描指针从第 0 个字符起步。
- `while i<len(command):` —— 只要还没扫到末尾就继续。i 严格递增（每轮至少 +1），所以循环一定终止。
- `if command.startswith('G',i):` / `pieces.append('G'); i+=1` —— 先问"从 i 起是不是字母 G"。是的话，把翻译 `'G'` 存进 pieces，指针**只前进 1**（token 长度）。注意这行用了分号把两条语句写在一行，是紧凑风格，语义和两行完全相同。
- `elif command.startswith('()',i):` / `pieces.append('o'); i+=2` —— 不是 G 再问"从 i 起是不是 `()`"。是就存 `'o'`，指针前进 2——括号整体被消费掉，`(` 和 `)` 都不会再被单独看到。
- `elif command.startswith('(al)',i):` / `pieces.append('al'); i+=4` —— 再问"从 i 起是不是 `(al)`"。是就存 `'al'`，指针前进 4。这里体现"跳过整个 token"的关键：吃掉 `(al)` 后 i 直接落在下一个 token 的开头，内部的 `al)` 不会被二次识别。
- `else:` / `raise ValueError(f'invalid token at index {i}')` —— 三个都问不上，说明从 i 开始的内容不属于任何合法 token，违反契约。抛 ValueError 并用 f-string 把出错下标写进错误消息，方便定位。
- `return ''.join(pieces)` —— 用空串当粘合剂把所有片段一次拼成结果。为什么不 `result += 片段`？因为字符串不可变，`+=` 每次都要复制整个已有结果再追加，循环里反复做就是 O(n²)；join 只做一次总装配，是 O(n)。

**小实例表格的生成代码（“运行示例”部分）：** “运行示例”部分 开头同样定义了 `show_table` 展示辅助函数（代码与上一章相同：escape 把值转成安全文本、拼成 HTML 表格、display 渲染，不含任何算法），实验部分就一行调用：

```python
show_table(['输入 token', '输出', '消费字符数'], [(token, interpret(token), len(token)) for token in ['G', '()', '(al)']])
```

- 列表推导式 `[(token,interpret(token),len(token)) for token in ['G','()','(al)']]` 让三个合法 token 轮流当 `token`，每轮产出一行三格：token 本身、**真的调用内核里的 `interpret`** 算出的翻译、以及 `len(token)` 给出的字符数。
- 第三列"消费字符数"用 `len(token)` 是有依据的：每个演示输入恰好只含一个 token，扫描会把它整个吃掉，所以"输入多长"就等于"指针前进了多远"。
- 逐格读三行：`'G'` 翻译成 `'G'`、消费 1 格；`'()'` 翻译成 `'o'`、消费 2 格；`'(al)'` 翻译成 `'al'`、消费 4 格。表格把"输出"和"步长"并排放着，就是在提醒你：这两件事由同一个 token 决定，翻译和前进格数必须配套——这正是上面代码里每个分支"append 什么"和"i+= 几"成对出现的原因。

## 5. 分段实现：按顺序运行

**本章接口：** `defang_ip(address)`、`interpret(command)`

### defang_ip

In [1]:
def defang_ip(address: str) -> str:
    return address.replace('.', '[.]')

### interpret

In [2]:
def interpret(command: str) -> str:
    pieces = []
    i = 0
    while i < len(command):
        if command.startswith('G', i):
            pieces.append('G')
            i += 1
        elif command.startswith('()', i):
            pieces.append('o')
            i += 2
        elif command.startswith('(al)', i):
            pieces.append('al')
            i += 4
        else:
            raise ValueError(f'invalid token at index {i}')
    return ''.join(pieces)

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

show_table(['输入 token','输出','消费字符数'], [(token,interpret(token),len(token)) for token in ['G','()','(al)']])

输入 token,输出,消费字符数
G,G,1
(),o,2
(al),al,4


## 7. 正确性、适用条件与复杂度

扫描指针前的字符都已被恰好一个合法 token 消费，pieces 是它们的正确翻译。三种 token 的起始结构可区分，每步前进保证终止。

**代价：** 时间 O(n)，输出与暂存片段 O(n)；join 避免反复拼接造成的复制。

### 展开理解

**为什么是对的：** 我们盯住扫描过程中始终成立的性质——"指针 i 左边的每个字符都恰好被某一个合法 token 整体消费掉了，pieces 里就是这些 token 的正确翻译按原顺序排好"。一开始 i=0，左边没有字符，性质空成立。每一轮：三种 if/elif 判断只会有一个命中（或者全不中直接抛错），命中的那个 token 从 i 开始完全匹配，我们把它的固定翻译追加进 pieces，并让 i 精确跳过它的长度——所以这轮消费的字符又被一个 token 恰好覆盖，性质保持。三种 token 的开头结构互相能区分（G 以字母开头，另外两个以 `(` 开头但第二字符一个是 `)` 一个是 `a`），所以从任何位置开始、输入合法时，判断不会认错 token。循环结束时 i 到达末尾，所有字符都被消费，pieces 拼起来就是整条命令的正确翻译。至于终止性：每轮 i 至少前进 1，len(command) 是固定上限，指针迟早越过末尾，循环必然停。

**复杂度是多少：** 时间 O(n)，n 是字符串长度。每个字符恰被一个 token 消费一次，startswith 每次只看常数个字符（最多 4 个）。输出和暂存片段加起来也是 O(n) 字符。拿具体数字感受：一条 10⁵ 个字符的命令（约 2.5 万个 `(al)`），扫描加拼装总共十几万次字符操作，一瞬间完成。join 一次装配避免了"循环里字符串 += 反复复制"的平方级陷阱，这正是它出现在这里的意义。

**用具体数字感受 join 的价值：** 假设一条命令的翻译结果总长 10⁵ 个字符、由约 2.5 万个片段组成。join 的做法是先确定总长、一次分配、一次填装，约 10⁵ 次字符写入就完事。`result += piece` 的做法是每拼一片都把已有的 result 整个复制一遍：按平均片段长 4 估算，第 k 次要复制约 4k 个字符，总计约 4×(1+2+…+25000) ≈ 12.5 亿次字符复制，比 join 多出四个数量级。练习 2 要你用计时实测的正是这个差距。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 逐条看：

```python
assert defang_ip('1.1.1.1') == '1[.]1[.]1[.]1'

assert interpret('G()(al)') == 'Goal'

assert interpret('') == ''

assert interpret('(al)G(al)()()G') == 'alGalooG'

try:
    interpret('(bad)')
except ValueError:
    pass
else:
    raise AssertionError('invalid token must fail explicitly')

print('N006: 所有本章断言通过')
```

- `assert defang_ip('1.1.1.1') == '1[.]1[.]1[.]1'`：正常值测试，也是 LeetCode 1108 的官方示例。四个点全部被替换、数字原样保留，验证 replace 的"全部替换"语义。
- `assert interpret('G()(al)') == 'Goal'`：三种 token 各出现一次的黄金样例（LeetCode 1678 示例）。它验证了 token 识别、翻译映射和指针步长三件事一次通过。
- `assert interpret('') == ''`：空串边界。空命令一次循环都不跑，join 空列表得空串，应该安静返回而不是报错。
- `assert interpret('(al)G(al)()()G') == 'alGalooG'`：长组合正常值。5 个 token 混合排列（al + G + al + o + o + G），专门考验指针在 4 长和 1 长 token 之间连续切换时不丢步。
- `try: interpret('(bad)')` / `except ValueError: pass` / `else: raise AssertionError(...)`：**负向测试**。`(bad)` 不是合法 token（`(` 后面既不是 `)` 也不是 `a`），函数必须抛 ValueError；try/except 接住异常算通过。`else` 分支的含义是"如果没抛异常反而算失败"——它逼着实现显式失败，不允许把非法输入默默翻译出错误输出。这是"快速失败"契约的自动化验收。
- 最后的 print：前面全部通过才会打印，代表整组测试通过。

**如果实现写错了，哪条断言会先响？** 做个故障推演，帮你建立"断言 = 报警器"的直觉：

- 把 `(al)` 分支的步长写成 `i+=1`：`interpret('G()(al)')` 在消费 `(al)` 后指针落在 `'a'` 上，三个 startswith 全不匹配，函数抛 ValueError，第二条断言以异常方式失败——步长写错往往不是"答错"，而是直接崩给你看。
- 把 `(al)` 的翻译写成 `'a'`（漏个 l）：第二条断言挂（`'Goa'` ≠ `'Goal'`），第四条也挂（`'aGaooG'` ≠ `'alGalooG'`），两条正路断言一起报警。
- 把 else 分支从抛错改成"跳过一个字符继续"：`(bad)` 的 5 个字符会被逐个跳完，函数返回空串而不抛异常，于是 try/except 的 `else` 分支触发 `AssertionError('invalid token must fail explicitly')`——负向测试专抓这种"静默吞错"的实现。

**把测试分个层：**

| 断言 | 层次 | 它守护的契约 |
| --- | --- | --- |
| `defang_ip('1.1.1.1') == '1[.]1[.]1[.]1'` | 正常值 | 全部替换、数字原样保留 |
| `interpret('G()(al)') == 'Goal'` | 正常值 | 三种 token 的识别、翻译、步长一次过 |
| `interpret('') == ''` | 边界 | 空输入安静返回 |
| `interpret('(al)G(al)()()G') == 'alGalooG'` | 长组合 | 长短 token 连续切换不丢步 |
| `(bad)` 必须抛 ValueError | 负向 | 非法输入显式失败 |

In [4]:
assert defang_ip('1.1.1.1') == '1[.]1[.]1[.]1'

assert interpret('G()(al)') == 'Goal'

assert interpret('') == ''

assert interpret('(al)G(al)()()G') == 'alGalooG'

try:
    interpret('(bad)')
except ValueError:
    pass
else:
    raise AssertionError('invalid token must fail explicitly')

print('N006: 所有本章断言通过')

N006: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 不用replace实现IP转换。

**练习 2：** 解释循环拼接与join的区别。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（不用 replace 实现 IP 转换）**：提示——手动模拟 replace：遍历 address 的每个字符，`c == '.'` 时往结果列表 append `'[.]'`，否则 append c，最后 `''.join` 拼回字符串。也可以按 `'.'` split 成四段再用 `'[.]'.join` 粘回去。手算示例用 `'1.1.1.1'`，期望 `'1[.]1[.]1[.]1'`；边界可用不含点的 `'255'`（原样返回）和空串（返回空串）。做完对比：你的手写循环和内置 replace 结果逐字符一致，就说明你真正理解了替换语义。
- **练习 2（解释循环拼接与 join 的区别）**：提示——核心差异在"字符串不可变"这一事实。`result += piece` 每次都要新建一个更长的字符串并把旧内容全部抄过去，做 k 次就是约 1+2+…+k 量级的字符复制，总代价平方级；`pieces.append` 只是在列表尾部加引用（廉价），最后 `join` 一次性按总长度分配并填装，总代价线性。验证方式：构造 10 万个片段，分别计时两种方式（`time.time()` 前后差），你会看到数量级差距；边界用单元素列表 `['x']` 确认 join 结果就是 `'x'`。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
def defang_ip(address: str) -> str:
    return address.replace('.', '[.]')

def interpret(command: str) -> str:
    pieces = []
    i = 0
    while i < len(command):
        if command.startswith('G', i):
            pieces.append('G')
            i += 1
        elif command.startswith('()', i):
            pieces.append('o')
            i += 2
        elif command.startswith('(al)', i):
            pieces.append('al')
            i += 4
        else:
            raise ValueError(f'invalid token at index {i}')
    return ''.join(pieces)

# 示例与回归测试
assert defang_ip('1.1.1.1') == '1[.]1[.]1[.]1'

assert interpret('G()(al)') == 'Goal'

assert interpret('') == ''

assert interpret('(al)G(al)()()G') == 'alGalooG'

try:
    interpret('(bad)')
except ValueError:
    pass
else:
    raise AssertionError('invalid token must fail explicitly')

print('N006: 所有本章断言通过')

N006: 所有本章断言通过


## 11. 代表题与迁移

- **1108. Defanging an IP Address**：IP 脱敏原题，练的是"局部替换"和字符串不可变下的 replace 语义，对应本章 `defang_ip`。
- **1678. Goal Parser Interpretation**：Goal Parser 原题，练的是"按 token 扫描 + 指针按 token 长度前进 + 用 join 组装"这套解析基本功，对应本章 `interpret`；学完后你对"为什么不能只逐字符判断括号"会有切身体会。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。